# Urban Congestion Index

Este notebook cria um modelo simples para o servico `urban-congestion-index` usando duas fontes:
- `traffic-flow`
- `traffic-speed`

Saidas do modelo:
- classe: `baixa`, `media`, `alta`
- indice: `congestion_index` (0 a 100)

---


In [4]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

FLOW_PATH = Path('../../L0/traffic-flow/output-example.txt').resolve()
SPEED_PATH = Path('../../L0/traffic-speed/output-example.txt').resolve()
MODEL_PATH = Path('model.joblib').resolve()

FLOW_PATH, SPEED_PATH, MODEL_PATH


(PosixPath('/home/makleyston/Projects/service-composition/services/L0/traffic-flow/output-example.txt'),
 PosixPath('/home/makleyston/Projects/service-composition/services/L0/traffic-speed/output-example.txt'),
 PosixPath('/home/makleyston/Projects/service-composition/services/L1/urban-congestion-index/model.joblib'))

In [5]:
def load_jsonl(path: Path) -> list[dict]:
    records = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records


def parse_flow(records: list[dict]) -> pd.DataFrame:
    rows = []
    for rec in records:
        payload = rec.get('urn:ufcity:payload', {})
        features = payload.get('features', {})
        result = rec.get('saref:hasResult', {})
        rows.append(
            {
                'flow_level': result.get('saref:hasValue'),
                'flow_confidence': result.get('urn:ufcity:confidence'),
                'flow_feature_of_interest': rec.get('saref:hasFeatureOfInterest'),
                'flow_id_eqp': features.get('ID EQP'),
                'flow_direction': features.get('SENTIDO'),
                'flow_lane': features.get('FAIXA'),
                'vehicle_count': features.get('vehicle_count'),
                'avg_speed': features.get('avg_speed'),
                'speed_std': features.get('speed_std'),
                'pct_moto': features.get('pct_moto'),
                'pct_heavy': features.get('pct_heavy'),
                'flow_hour': features.get('hour'),
                'flow_weekday': features.get('weekday'),
            }
        )
    return pd.DataFrame(rows)


def extract_road_segment(feature_of_interest: str | None) -> float:
    if not feature_of_interest or not isinstance(feature_of_interest, str):
        return np.nan
    parts = feature_of_interest.split(':')
    if not parts:
        return np.nan
    try:
        return float(parts[-1])
    except ValueError:
        return np.nan


def parse_speed(records: list[dict]) -> pd.DataFrame:
    rows = []
    for rec in records:
        result = rec.get('saref:hasResult', {})
        feature_of_interest = rec.get('saref:hasFeatureOfInterest')
        ts = rec.get('saref:hasTimestamp', '')
        dt = pd.to_datetime(ts, errors='coerce')
        rows.append(
            {
                'speed_level': result.get('saref:hasValue'),
                'speed_confidence': result.get('urn:ufcity:confidence'),
                'speed_feature_of_interest': feature_of_interest,
                'speed_road_segment': extract_road_segment(feature_of_interest),
                'speed_hour': dt.hour if not pd.isna(dt) else np.nan,
                'speed_weekday': dt.dayofweek if not pd.isna(dt) else np.nan,
            }
        )
    return pd.DataFrame(rows)


flow_df = parse_flow(load_jsonl(FLOW_PATH))
speed_df = parse_speed(load_jsonl(SPEED_PATH))

flow_df.shape, speed_df.shape


((5819, 13), (3681, 6))

In [6]:
# Combinacao sintetica para PoC (nao existe alinhamento temporal real entre as fontes)
rng = np.random.default_rng(42)
sample_size = min(len(flow_df), len(speed_df))

flow_sample = flow_df.sample(n=sample_size, random_state=42, replace=False).reset_index(drop=True)
speed_sample = speed_df.sample(n=sample_size, random_state=42, replace=False).reset_index(drop=True)

df = pd.concat([flow_sample, speed_sample], axis=1)

numeric_cols = [
    'flow_confidence', 'flow_id_eqp', 'flow_lane', 'vehicle_count', 'avg_speed', 'speed_std',
    'pct_moto', 'pct_heavy', 'flow_hour', 'flow_weekday',
    'speed_confidence', 'speed_road_segment', 'speed_hour', 'speed_weekday'
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

speed_map = {'livre': 20, 'moderado': 60, 'congestionado': 90}
flow_map = {'baixo': 20, 'moderado': 60, 'alto': 90}

speed_score = df['speed_level'].map(speed_map).fillna(50)
flow_score = df['flow_level'].map(flow_map).fillna(50)
conf_mean = df[['speed_confidence', 'flow_confidence']].mean(axis=1).fillna(0.5)

rule_score = (0.6 * speed_score + 0.4 * flow_score)
rule_score = rule_score * (0.7 + 0.3 * conf_mean)
df['congestion_rule_score'] = np.clip(rule_score, 0, 100)

df['urban_congestion_level'] = pd.cut(
    df['congestion_rule_score'],
    bins=[-1, 40, 70, 101],
    labels=['baixa', 'media', 'alta'],
).astype('object')

df['urban_congestion_level'].value_counts(normalize=True).rename('proporcao')


urban_congestion_level
baixa    0.853301
media    0.146156
alta     0.000543
Name: proporcao, dtype: float64

In [7]:
feature_cols = [
    'flow_level', 'flow_confidence', 'flow_feature_of_interest', 'flow_direction', 'flow_lane',
    'vehicle_count', 'avg_speed', 'speed_std', 'pct_moto', 'pct_heavy', 'flow_hour', 'flow_weekday',
    'speed_level', 'speed_confidence', 'speed_feature_of_interest', 'speed_road_segment', 'speed_hour', 'speed_weekday'
]
target_col = 'urban_congestion_level'

X = df[feature_cols]
y = df[target_col]

numeric_features = [
    'flow_confidence', 'flow_lane', 'vehicle_count', 'avg_speed', 'speed_std', 'pct_moto',
    'pct_heavy', 'flow_hour', 'flow_weekday', 'speed_confidence', 'speed_road_segment',
    'speed_hour', 'speed_weekday'
]
categorical_features = [
    'flow_level', 'flow_feature_of_interest', 'flow_direction',
    'speed_level', 'speed_feature_of_interest'
]

numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

clf = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        (
            'model',
            RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                class_weight='balanced',
                n_jobs=-1,
            ),
        ),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

clf.fit(X_train, y_train)
pred = clf.predict(X_test)

print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))


              precision    recall  f1-score   support

       baixa       1.00      0.98      0.99       629
       media       0.89      1.00      0.94       108

    accuracy                           0.98       737
   macro avg       0.94      0.99      0.96       737
weighted avg       0.98      0.98      0.98       737

[[615  14]
 [  0 108]]


In [8]:
index_weights = {'baixa': 0.0, 'media': 50.0, 'alta': 100.0}
classes = list(clf.named_steps['model'].classes_)
class_weight_vector = np.array([index_weights.get(c, 50.0) for c in classes], dtype=float)

proba = clf.predict_proba(X_test.head(10))
congestion_index = np.clip(np.dot(proba, class_weight_vector), 0, 100)

preview = X_test.head(10).copy()
preview['nivel_real'] = y_test.head(10).values
preview['nivel_predito'] = clf.predict(X_test.head(10))
preview['congestion_index'] = np.round(congestion_index, 2)
preview


,flow_level,flow_confidence,flow_feature_of_interest,flow_direction,flow_lane,vehicle_count,avg_speed,speed_std,pct_moto,pct_heavy,...,flow_weekday,speed_level,speed_confidence,speed_feature_of_interest,speed_road_segment,speed_hour,speed_weekday,nivel_real,nivel_predito,congestion_index
1432,baixo,0.945,urn:ufcity:traffic:segment:193:centro-barro-pr...,Centro/Barro Preto,3,22,51.454545,5.324939,0.045455,0.227273,...,4,livre,0.833667,urn:ufcity:traffic:road-segment:390,390.0,0,4,baixa,baixa,0.00
2469,moderado,0.580,urn:ufcity:traffic:segment:198:centro-bairro:3,Centro/Bairro,3,54,50.629630,4.957542,0.222222,0.055556,...,4,livre,0.583632,urn:ufcity:traffic:road-segment:368,368.0,0,4,baixa,baixa,0.50
1930,baixo,0.805,urn:ufcity:traffic:segment:242:centro-bairro:2,Centro/Bairro,2,2,60.500000,3.535534,0.500000,0.500000,...,4,livre,0.764534,urn:ufcity:traffic:road-segment:363,363.0,0,4,baixa,baixa,0.83
3005,baixo,0.990,urn:ufcity:traffic:segment:226:savassi-sta-efi...,Savassi / Sta. Efigênia,1,6,50.833333,5.879342,0.166667,0.000000,...,4,livre,0.749175,urn:ufcity:traffic:road-segment:336,336.0,0,4,baixa,baixa,0.17
2847,baixo,0.840,urn:ufcity:traffic:segment:225:pra-a-da-esta-o...,Praça da Estação / Rodoviária,2,44,45.090909,6.068670,0.090909,0.090909,...,4,livre,0.711263,urn:ufcity:traffic:road-segment:360,360.0,0,4,baixa,baixa,0.33
106,baixo,0.990,urn:ufcity:traffic:segment:214:bairro-centro:1,Bairro/Centro,1,5,52.600000,5.412947,0.200000,0.000000,...,4,livre,0.511246,urn:ufcity:traffic:road-segment:315,315.0,0,4,baixa,baixa,0.33
2952,baixo,0.990,urn:ufcity:traffic:segment:207:bairro-centro:3,Bairro/Centro,3,29,48.655172,6.980973,0.068966,0.137931,...,4,livre,0.721680,urn:ufcity:traffic:road-segment:393,393.0,0,4,baixa,baixa,0.00
1386,baixo,0.965,urn:ufcity:traffic:segment:207:bairro-centro:2,Bairro/Centro,2,6,41.666667,7.685484,0.166667,0.166667,...,4,moderado,0.588613,urn:ufcity:traffic:road-segment:354,354.0,0,4,media,media,43.33
1600,baixo,0.935,urn:ufcity:traffic:segment:199:centro-bairro:2,Centro/Bairro,2,10,46.500000,2.321398,0.200000,0.000000,...,4,livre,0.778378,urn:ufcity:traffic:road-segment:331,331.0,0,4,baixa,baixa,0.17
923,baixo,0.535,urn:ufcity:traffic:segment:225:pra-a-da-esta-o...,Praça da Estação / Rodoviária,2,61,44.868852,5.475629,0.081967,0.081967,...,4,livre,0.576351,urn:ufcity:traffic:road-segment:391,391.0,0,4,baixa,baixa,0.17


In [9]:
artifact = {
    'pipeline': clf,
    'features': feature_cols,
    'target': target_col,
    'classes': classes,
    'index_name': 'congestion_index',
    'index_scale': [0, 100],
    'index_weights': index_weights,
    'labeling_rule': {
        'score_formula': 'score = (0.6*speed + 0.4*flow) * (0.7 + 0.3*mean_confidence)',
        'baixa': 'score <= 40',
        'media': '40 < score <= 70',
        'alta': 'score > 70',
    },
}

joblib.dump(artifact, MODEL_PATH)
MODEL_PATH


PosixPath('/home/makleyston/Projects/service-composition/services/L1/urban-congestion-index/model.joblib')